 Importing Libraries

In [34]:
import pandas as pd
import numpy as np
from prophet import Prophet
from tqdm import tqdm
from functools import reduce
from sklearn.model_selection import train_test_split
from xgboost import XGBRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.svm import SVR
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.ensemble import VotingRegressor
import joblib


Loading the Data

In [16]:
df=pd.read_csv("../ISI_dataset\merged_gram_reservoir.csv")
df.head()

<>:1: SyntaxWarning: invalid escape sequence '\m'
<>:1: SyntaxWarning: invalid escape sequence '\m'
C:\Users\shrey\AppData\Local\Temp\ipykernel_20764\776255179.py:1: SyntaxWarning: invalid escape sequence '\m'
  df=pd.read_csv("../ISI_dataset\merged_gram_reservoir.csv")


,state_name,crop_name,apy_item_interval_start,temperature_recorded_date,state_temperature_max_val,state_temperature_min_val,state_rainfall_val,yield,FRL,Live Cap FRL,Level,Current Live Storage
0,Andhra Pradesh,gram,2000,2000-01-01,30.38,14.47,0.0,1.22615,152.296667,2.838333,266.30,6.390
1,Andhra Pradesh,gram,2000,2000-01-02,30.04,13.96,0.0,1.22615,152.296667,2.838333,266.18,6.330
2,Andhra Pradesh,gram,2000,2000-01-03,29.92,12.98,0.0,1.22615,152.296667,2.838333,266.09,6.286
3,Andhra Pradesh,gram,2000,2000-01-04,29.98,12.23,0.0,1.22615,152.296667,2.838333,266.03,6.257
4,Andhra Pradesh,gram,2000,2000-01-05,29.77,13.24,0.0,1.22615,152.296667,2.838333,265.97,6.228


Changing type of date to datetime and renaming year

In [17]:
df['temperature_recorded_date'] = pd.to_datetime(df['temperature_recorded_date'])
df['year'] = df['temperature_recorded_date'].dt.year
# Drop unreliable states
df = df[~df['state_name'].isin(['Odisha', 'Tamil Nadu',])]

Droping unreliable states because there is not enough years of data

In [18]:
# Use only data till 2022 for training
df = df[df['year'] < 2023].copy()

# Group annually to match 2023 structure
df_annual = df.groupby(['state_name', 'crop_name', 'year']).agg({
    'state_rainfall_val': 'sum',
    'state_temperature_max_val': 'mean',
    'state_temperature_min_val': 'mean',
    'Live Cap FRL': 'mean',
    'FRL': 'mean',
    'Level': 'mean',
    'Current Live Storage': 'mean',
    'yield': 'mean'
}).reset_index()

In [19]:
df_annual.head()

,state_name,crop_name,year,state_rainfall_val,state_temperature_max_val,state_temperature_min_val,Live Cap FRL,FRL,Level,Current Live Storage,yield
0,Andhra Pradesh,gram,2000,942.03,34.738197,18.885082,2.838333,152.296667,179.065178,2.636298,1.22615
1,Andhra Pradesh,gram,2001,918.85,35.137890,18.995397,2.838333,152.296667,171.736301,1.820863,1.30074
2,Andhra Pradesh,gram,2002,654.57,35.292247,18.858493,2.838333,152.296667,169.014014,1.402067,0.93353
3,Andhra Pradesh,gram,2003,832.53,35.550356,19.248548,2.838333,152.296667,161.787795,0.781289,1.05206
4,Andhra Pradesh,gram,2004,786.89,34.954836,18.399536,2.838333,152.296667,164.511298,1.314224,1.04802


In [20]:
# One-hot encode 'state_name'
df_encoded = pd.get_dummies(df_annual, columns=['state_name'])

# Define features: original + one-hot encoded state columns
state_columns = [col for col in df_encoded.columns if col.startswith('state_name_')]

Train test split (till 2020 is taken as training data and years 2021 & 2022 are in testing data)

In [21]:
# Define features and target
features = ['state_rainfall_val', 'state_temperature_max_val', 'state_temperature_min_val', 'Live Cap FRL', 'FRL','Level','Current Live Storage']+ state_columns

# Split manually by year
train_df = df_encoded[df_encoded['year'] <= 2020]
test_df = df_encoded[df_encoded['year'].between(2021, 2022)]

In [22]:
X_train = train_df[features]
y_train = train_df['yield']
X_test = test_df[features]
y_test = test_df['yield']
X_train.head()

,state_rainfall_val,state_temperature_max_val,state_temperature_min_val,Live Cap FRL,FRL,Level,Current Live Storage,state_name_Andhra Pradesh,state_name_Chhattisgarh,state_name_Gujarat,state_name_Jharkhand,state_name_Karnataka,state_name_Madhya Pradesh,state_name_Maharashtra,state_name_Rajasthan,state_name_Telangana,state_name_Uttar Pradesh,state_name_Uttarakhand,state_name_West Bengal
0,942.03,34.738197,18.885082,2.838333,152.296667,179.065178,2.636298,True,False,False,False,False,False,False,False,False,False,False,False
1,918.85,35.137890,18.995397,2.838333,152.296667,171.736301,1.820863,True,False,False,False,False,False,False,False,False,False,False,False
2,654.57,35.292247,18.858493,2.838333,152.296667,169.014014,1.402067,True,False,False,False,False,False,False,False,False,False,False,False
3,832.53,35.550356,19.248548,2.838333,152.296667,161.787795,0.781289,True,False,False,False,False,False,False,False,False,False,False,False
4,786.89,34.954836,18.399536,2.838333,152.296667,164.511298,1.314224,True,False,False,False,False,False,False,False,False,False,False,False


In [23]:
# Models to compare
models = {
    'Linear Regression': LinearRegression(),
    'Random Forest': RandomForestRegressor(random_state=42),
    'XGBoost': XGBRegressor(random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(random_state=42),
    'Support Vector Regressor': SVR()
}

# Results container
results = []

# Loop through models
for name, model in models.items():
    model.fit(X_train, y_train)

    train_pred = model.predict(X_train)
    test_pred = model.predict(X_test)

    # Metrics
    train_r2 = r2_score(y_train, train_pred)
    test_r2 = r2_score(y_test, test_pred)

    train_rmse = np.sqrt(mean_squared_error(y_train, train_pred))
    test_rmse = np.sqrt(mean_squared_error(y_test, test_pred))

    results.append({
        'Model': name,
        'Train R²': round(train_r2, 4),
        'Test R²': round(test_r2, 4),
        'Train RMSE': round(train_rmse, 2),
        'Test RMSE': round(test_rmse, 2)
    })

# Display results
results_df = pd.DataFrame(results)
print(results_df.sort_values(by='Test R²', ascending=False))

                      Model  Train R²  Test R²  Train RMSE  Test RMSE
1             Random Forest    0.9279   0.3818        0.08       0.26
2                   XGBoost    1.0000   0.3628        0.00       0.26
3         Gradient Boosting    0.9145   0.2336        0.09       0.29
0         Linear Regression    0.5297   0.1323        0.20       0.31
4  Support Vector Regressor    0.3052  -0.1389        0.24       0.35


Incoperating our top 2 or 3 best perfoming models

In [30]:
# Initialize individual models
rf = RandomForestRegressor()
xgb = XGBRegressor(random_state=42)

# Ensemble model
ensemble = VotingRegressor(estimators=[
    ('rf', rf),
    ('xgb', xgb)
])

# Fit ensemble
ensemble.fit(X_train, y_train)

# Predict
train_pred = ensemble.predict(X_train)
test_pred = ensemble.predict(X_test)

# Evaluate
train_r2 = r2_score(y_train, train_pred)
test_r2 = r2_score(y_test, test_pred)
train_rmse = np.sqrt(mean_squared_error(y_train, train_pred))
test_rmse = np.sqrt(mean_squared_error(y_test, test_pred))

print("📊 Ensemble Performance:")
print(f"Train R²: {train_r2:.4f}, Test R²: {test_r2:.4f}")
print(f"Train RMSE: {train_rmse:.4f}, Test RMSE: {test_rmse:.4f}")


📊 Ensemble Performance:
Train R²: 0.9816, Test R²: 0.3769
Train RMSE: 0.0395, Test RMSE: 0.2609


In [35]:
joblib.dump(ensemble,'../models/gram.pkl')

['../models/gram.pkl']

## Predicting the environment parameters of year 2023 using Prophet TSA model

In [25]:
# --- 1. Define function to forecast any single feature using Prophet ---
def forecast_feature_prophet(df, feature_name):
    forecast_data = []

    for (state, crop), group in tqdm(df.groupby(['state_name', 'crop_name'])):
        yearly_data = group.groupby('year')[feature_name].mean().reset_index()

        if yearly_data.shape[0] < 4:
            continue

        prophet_df = yearly_data.rename(columns={'year': 'ds', feature_name: 'y'})
        prophet_df['ds'] = pd.to_datetime(prophet_df['ds'], format='%Y')

        try:
            model = Prophet()
            model.fit(prophet_df)

            future = pd.DataFrame({'ds': [pd.to_datetime('2023')]})
            forecast = model.predict(future)
            yhat = forecast['yhat'].values[0]

            forecast_data.append({
                'state_name': state,
                'crop_name': crop,
                feature_name: yhat
            })
        except:
            continue

    return pd.DataFrame(forecast_data)

# --- 2. Forecast each feature separately ---
df_rain = forecast_feature_prophet(df, 'state_rainfall_val')
df_temp_max = forecast_feature_prophet(df, 'state_temperature_max_val')
df_temp_min = forecast_feature_prophet(df, 'state_temperature_min_val')
df_livecap = forecast_feature_prophet(df, 'Live Cap FRL')
df_frl = forecast_feature_prophet(df, 'FRL')
df_level = forecast_feature_prophet(df, 'Level')
df_cls = forecast_feature_prophet(df, 'Current Live Storage')

# --- 3. Merge all forecasted dataframes ---
from functools import reduce
dfs = [df_rain, df_temp_max, df_temp_min, df_livecap, df_frl, df_level, df_cls]
df_2023 = reduce(lambda left, right: pd.merge(left, right, on=['state_name', 'crop_name'], how='outer'), dfs)

# --- 4. One-hot encode state_name ---
df_2023_encoded = df_2023.copy()  # Keep original columns
state_names = df_2023_encoded[['state_name', 'crop_name']]  # Keep for merging later

df_2023_encoded = pd.get_dummies(df_2023_encoded, columns=['state_name'])
df_2023_encoded = pd.concat([state_names, df_2023_encoded.drop(columns=['crop_name'])], axis=1)



  0%|          | 0/12 [00:00<?, ?it/s]

01:21:17 - cmdstanpy - INFO - Chain [1] start processing
01:21:18 - cmdstanpy - INFO - Chain [1] done processing
  8%|▊         | 1/12 [00:00<00:08,  1.28it/s]01:21:18 - cmdstanpy - INFO - Chain [1] start processing
01:21:18 - cmdstanpy - INFO - Chain [1] done processing
 17%|█▋        | 2/12 [00:00<00:04,  2.27it/s]01:21:18 - cmdstanpy - INFO - Chain [1] start processing
01:21:18 - cmdstanpy - INFO - Chain [1] done processing
 25%|██▌       | 3/12 [00:01<00:03,  2.71it/s]01:21:18 - cmdstanpy - INFO - Chain [1] start processing
01:21:18 - cmdstanpy - INFO - Chain [1] done processing
 33%|███▎      | 4/12 [00:01<00:02,  3.05it/s]01:21:18 - cmdstanpy - INFO - Chain [1] start processing
01:21:19 - cmdstanpy - INFO - Chain [1] done processing
 42%|████▏     | 5/12 [00:01<00:02,  2.77it/s]01:21:19 - cmdstanpy - INFO - Chain [1] start processing
01:21:19 - cmdstanpy - INFO - Chain [1] done processing
 50%|█████     | 6/12 [00:02<00:02,  2.89it/s]01:21:19 - cmdstanpy - INFO - Chain [1] start 

In [26]:
df_2023_encoded.head()

,state_name,crop_name,state_rainfall_val,state_temperature_max_val,state_temperature_min_val,Live Cap FRL,FRL,Level,Current Live Storage,state_name_Andhra Pradesh,...,state_name_Gujarat,state_name_Jharkhand,state_name_Karnataka,state_name_Madhya Pradesh,state_name_Maharashtra,state_name_Rajasthan,state_name_Telangana,state_name_Uttar Pradesh,state_name_Uttarakhand,state_name_West Bengal
0,Andhra Pradesh,gram,2.790352,34.392923,19.347568,2.179976,198.326146,142.472016,0.996292,True,...,False,False,False,False,False,False,False,False,False,False
1,Chhattisgarh,gram,3.704114,33.805702,18.200942,1.365667,377.820000,353.377518,1.355013,False,...,False,False,False,False,False,False,False,False,False,False
2,Gujarat,gram,2.706400,34.828181,18.653730,0.811491,115.199781,119.286907,0.366662,False,...,True,False,False,False,False,False,False,False,False,False
3,Jharkhand,gram,3.394479,33.182895,18.405240,0.404750,296.267500,290.707379,0.200885,False,...,False,True,False,False,False,False,False,False,False,False
4,Karnataka,gram,3.444745,33.812358,17.518092,1.539062,597.731875,586.334229,0.790733,False,...,False,False,True,False,False,False,False,False,False,False


In [31]:
X_2023 = df_2023_encoded[features]

# Use your trained ensemble model
y_2023_pred = ensemble.predict(X_2023)

# Add prediction to the dataframe
df_2023_encoded['predicted_yield'] = y_2023_pred

# Select output
output_2023 = df_2023_encoded[['crop_name'] + [col for col in df_2023_encoded.columns if col.startswith('state_name_')] + ['predicted_yield']]


In [ ]:
# Convert dummy columns back to state_name
state_names = df_2023_encoded[[col for col in df_2023_encoded.columns if col.startswith('state_name_')]].idxmax(axis=1)
state_names = state_names.str.replace('state_name_', '')

# Final output
final_2023_yield = pd.DataFrame({
    'state_name': state_names,
    'crop_name': df_2023_encoded['crop_name'],
    'predicted_yield_2023': df_2023_encoded['predicted_yield'].round(2)
})

print(final_2023_yield)
# final_2023_yield.to_csv("../yield_prediction.csv", mode='a', header=False, index=False)

        state_name crop_name  predicted_yield_2023
0   Andhra Pradesh      gram                  0.69
1     Chhattisgarh      gram                  0.58
2          Gujarat      gram                  0.78
3        Jharkhand      gram                  0.98
4        Karnataka      gram                  0.60
5   Madhya Pradesh      gram                  0.98
6      Maharashtra      gram                  0.64
7        Rajasthan      gram                  0.56
8        Telangana      gram                  1.15
9    Uttar Pradesh      gram                  0.54
10     Uttarakhand      gram                  0.67
11     West Bengal      gram                  0.68


Comparing our predicted yield with previous year yields

In [33]:
# Step 1: Get actual yields from 2019 to 2022
df_recent = df_annual[df_annual['year'].between(2019, 2022)].copy()

# Pivot to get each year's yield as a column
yield_table = df_recent.pivot_table(
    index=['state_name', 'crop_name'],
    columns='year',
    values='yield'
).reset_index()

# Rename columns for clarity
yield_table = yield_table.rename(columns={
    2019: 'yield_2019',
    2020: 'yield_2020',
    2021: 'yield_2021',
    2022: 'yield_2022'
})

# Step 2: Prepare 2023 predicted yield
df_2023_yield = df_2023_encoded[['state_name', 'crop_name', 'predicted_yield']].copy()
df_2023_yield = df_2023_yield.rename(columns={'predicted_yield': 'yield_2023'})

# Step 3: Merge the 2023 predicted yield into the table
final_yield_table = pd.merge(yield_table, df_2023_yield, on=['state_name', 'crop_name'], how='left')

# Display final table
print(final_yield_table)


        state_name crop_name  yield_2019  yield_2020  yield_2021  yield_2022  \
0   Andhra Pradesh      gram     1.21843     1.13584     1.02136     1.42080   
1     Chhattisgarh      gram     0.25200     0.82200     0.78200     0.89800   
2          Gujarat      gram     1.57136     1.76172     1.90761     1.69906   
3        Jharkhand      gram     1.19747     1.25740     1.18552     1.17171   
4        Karnataka      gram     0.74260     0.62472     0.68892     0.67990   
5   Madhya Pradesh      gram     1.59054     1.66188     1.51444     1.69011   
6      Maharashtra      gram     1.09636     1.07437     1.14457     1.01340   
7        Rajasthan      gram     1.07926     1.04518     1.16674     0.93361   
8        Telangana      gram     1.53178     1.66746     1.40915     1.56799   
9    Uttar Pradesh      gram     1.37120     1.24336     1.34588     1.34927   
10     Uttarakhand      gram     0.75762     0.76648     0.80597     0.79952   
11     West Bengal      gram     1.31344

<!-- leakage-safe-evaluation -->
## Leakage-safe evaluation

This section evaluates the **gram** yield model as a rolling one-year-ahead
forecast. Models are selected using five-year walk-forward validation on data
through 2020 and evaluated once on the untouched 2021-2022 holdout.

- **Previous-year baseline:** predicts the current yield from the previous
  calendar year's observed yield.
- **Lagged features:** previous-year yield and the trailing three-year mean;
  both use only earlier targets.
- **WAPE:** total absolute error divided by total actual yield. Lower is better.
- **Normalized RMSE:** RMSE divided by mean absolute yield. Lower is better.

Observed weather and reservoir variables are used for the evaluation year, so
these results evaluate the yield-regression stage rather than the complete
Prophet-to-yield forecasting pipeline.

In [1]:
from pathlib import Path
import os
import sys
from IPython.display import display
import pandas as pd

repo_root = Path.cwd().resolve()
while not (repo_root / "evaluate_models.py").is_file():
    if repo_root.parent == repo_root:
        raise FileNotFoundError("Could not locate evaluate_models.py")
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from evaluate_models import (  # noqa: E402
    DATASETS,
    build_models,
    evaluate_dataset,
    prepare_annual_data,
)

crop_key = "gram"
spec = DATASETS[crop_key]
data_dir = Path(os.environ.get("ISI_DATA_DIR", repo_root / "ISI_dataset"))
annual_evaluation_data = prepare_annual_data(
    data_dir / spec.filename, spec.excluded_states
)

In [2]:
evaluation_summary, fold_results, cv_results, holdout_predictions = evaluate_dataset(
    crop_key,
    annual_evaluation_data,
    build_models(),
    holdout_start=2021,
    holdout_end=2022,
    cv_years=5,
    min_train_years=5,
)

print("Walk-forward validation (sorted by RMSE):")
display(
    cv_results[["model", "folds", "r2", "mae", "rmse", "nrmse_pct", "wape_pct"]]
    .round(4)
)

print("2021-2022 holdout result:")
display(
    pd.DataFrame([evaluation_summary])[
        [
            "selected_model",
            "holdout_samples",
            "holdout_r2",
            "holdout_mae",
            "holdout_rmse",
            "holdout_nrmse_pct",
            "holdout_wape_pct",
            "baseline_rmse",
            "rmse_improvement_vs_baseline_pct",
        ]
    ].round(4)
)

Walk-forward validation (sorted by RMSE):


,model,folds,r2,mae,rmse,nrmse_pct,wape_pct
0,SVR,5,0.4144,0.1912,0.2389,21.3770,17.1065
1,Linear Regression,5,0.3262,0.2090,0.2562,22.9291,18.6984
2,Random Forest,5,0.3129,0.1977,0.2587,23.1543,17.6935
3,Gradient Boosting,5,0.3016,0.2047,0.2609,23.3449,18.3169
4,Voting Ensemble,5,0.3003,0.2053,0.2611,23.3651,18.3721
5,XGBoost,5,0.1584,0.2156,0.2864,25.6257,19.2959


2021-2022 holdout result:


,selected_model,holdout_samples,holdout_r2,holdout_mae,holdout_rmse,holdout_nrmse_pct,holdout_wape_pct,baseline_rmse,rmse_improvement_vs_baseline_pct
0,SVR,24.0,0.8578,0.1033,0.1246,10.4297,8.6435,0.1468,15.1067
